# Work places: numbers, distances, maps, timing

Visual companion to `sql/07_work.sql` (rules in `DECISIONS.md`, D19). A work place is a minimal label, set only where
it is clear: the res 10 cell outside the home area (home cell + ring 1) with the most distinct Mon-Fri dates with a
tweet at 09-16h local, if it has >= 3 such dates and strictly more than any other cell. Most users get none.

| section | reads (schema `revised`) |
|---|---|
| 1. Numbers | `home_final`, `work_final` |
| 2. Commute distance | `work_final` |
| 3. Maps | `work_final`, `home_final` |
| 4. Timing | `visit_stay`, `visit_home_area`, `work_final` |

Automated accounts (`visit_user.automated`, 08 S3: >= 10 visits per visit day) are left out everywhere (`WORK` in
the helpers cell).

Credentials: `code/03_database_setup/connection.json`.

In [ ]:
# ── imports ───────────────────────────────────────────────────────────────────
import json
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from matplotlib.colors import LogNorm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy.stats import spearmanr
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

FS = 10
FS2 = FS - 2
FS4 = FS - 4
plt.rcParams.update({"font.size": FS})

# ── connection ────────────────────────────────────────────────────────────────
cfg = json.load(open(Path("../../03_database_setup/connection.json")))
url = URL.create(
    "postgresql+psycopg2",
    username=cfg["user"],
    password=cfg["password"],
    host=cfg["host"],
    port=cfg["port"],
    database=cfg["database"],
)
# public on the path for the h3index operators and functions (D14)
engine = create_engine(url, connect_args={"options": "-c search_path=revised,public"})

# ── cities: order, labels, colours (fixed per city in every figure) ───────────
CITIES = ["amsterdam", "london", "portland"]
CITY_LABEL = {"amsterdam": "Amsterdam", "london": "London", "portland": "Portland"}
CITY_COLOR = {"amsterdam": "#2a78d6", "london": "#eb6834", "portland": "#1baf7a"}
PANEL = ["(A)", "(B)", "(C)", "(D)"]

# ── zoom windows for the maps (lon_min, lat_min, lon_max, lat_max) ────────────
ZOOM = {
    "amsterdam": (4.84, 52.345, 4.95, 52.395),
    "london":    (-0.20, 51.48, 0.00, 51.56),
    "portland":  (-122.72, 45.49, -122.61, 45.56),
}

In [ ]:
# ── helpers ───────────────────────────────────────────────────────────────────
def style_axis(ax, label=None, grid=True):
    """Clean spines, small ticks, bold panel label."""
    ax.tick_params(labelsize=FS2)
    ax.spines[["top", "right"]].set_visible(False)
    if grid:
        ax.grid(axis="y", color="0.9", linewidth=0.8)
        ax.set_axisbelow(True)
    if label:
        ax.text(0.0, 1.03, label, transform=ax.transAxes, fontsize=FS, fontweight="bold")


def read_hex(sql):
    """Query with an h3 `cell` column -> GeoDataFrame of hexagons in EPSG:3857."""
    gdf = gpd.read_postgis(
        f"select q.*, public.h3_cell_to_boundary_geometry(q.cell) as geom from ({sql}) q",
        engine, geom_col="geom", crs="EPSG:4326",
    )
    gdf = gdf.rename_geometry("geometry")
    gdf["cell"] = gdf["cell"].astype(str)
    return gdf.to_crs(epsg=3857)


def zoom_extent(city):
    """Zoom window of a city in EPSG:3857."""
    lon0, lat0, lon1, lat1 = ZOOM[city]
    pts = gpd.GeoSeries.from_xy([lon0, lon1], [lat0, lat1], crs="EPSG:4326").to_crs(epsg=3857)
    return pts.x.min(), pts.x.max(), pts.y.min(), pts.y.max()


def hex_map(ax, gdf, column, norm, cbar_label, label, extent=None, cmap="Blues"):
    """Hexagons coloured by `column` on a light basemap, colorbar matched to the axis."""
    gdf.plot(column=column, cmap=cmap, norm=norm, linewidth=0, alpha=0.85, ax=ax)
    if extent is not None:
        ax.set_xlim(extent[0], extent[1])
        ax.set_ylim(extent[2], extent[3])
    ctx.add_basemap(ax, source=ctx.providers.Esri.WorldGrayCanvas, attribution_size=FS4)
    ax.set_axis_off()
    ax.text(0.0, 1.02, label, transform=ax.transAxes, fontsize=FS, fontweight="bold")
    cax = make_axes_locatable(ax).append_axes("right", size="4%", pad=0.05)
    cb = plt.gcf().colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), cax=cax)
    cb.set_label(cbar_label, fontsize=FS2)
    cb.ax.tick_params(labelsize=FS2)


# ── automated accounts (08 S3, revised.visit_user.automated) are left out ─────
VISIT = """(select v.* from revised.visit v
         join revised.visit_user u on u.city = v.city and u.user_id = v.user_id
         where not u.automated)"""
WORK = """(select w.* from revised.work_final w
        join revised.visit_user u on u.city = w.city and u.user_id = w.user_id
        where not u.automated)"""

---
# 1. Numbers
## 1.1 Users with a work place

In [ ]:
# ── summary per city ──────────────────────────────────────────────────────────
work_sum = pd.read_sql(f"""
select u.city
     , count(*)                                                       as n_home_users
     , count(w.user_id)                                               as n_work
     , round(avg((w.user_id is not null)::int)::numeric, 3)           as share_work
     , percentile_cont(0.5) within group (order by w.d_work_days)     as median_work_days
     , percentile_cont(0.5) within group (order by w.dist_home_m)     as median_dist_m
from (select distinct city, user_id from revised.home_final) u
left join {WORK} w
  on w.city = u.city and w.user_id = u.user_id
group by u.city
order by u.city
""", engine)
work_sum

## 1.2 Evidence: weekdays in the work cell

Distinct Mon-Fri dates with a daytime tweet in the work cell, and in the user's second cell. The gate is 3 days and
a strict lead.

In [ ]:
# ── work days and runner-up per work user ─────────────────────────────────────
wdays = pd.read_sql(f"""
select city, d_work_days, coalesce(d_second_days, 0) as d_second_days, dist_home_m
from {WORK} w
""", engine)

# ── plot: (A) work days, (B) work days vs second cell ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
bins = np.arange(3, 41)
for city in CITIES:
    d = wdays.loc[wdays["city"] == city, "d_work_days"].clip(upper=40)
    axes[0].hist(d, bins=bins, density=True, histtype="step", linewidth=1.8,
                 color=CITY_COLOR[city], label=CITY_LABEL[city])
axes[0].set_xlabel("weekdays with daytime tweets in the work cell (40 = 40+)", fontsize=FS2)
axes[0].set_ylabel("share of work users", fontsize=FS2)
axes[0].legend(frameon=False, fontsize=FS2)
style_axis(axes[0], "(A)")

for city in CITIES:
    d = wdays[wdays["city"] == city]
    lead = (d["d_work_days"] - d["d_second_days"]).clip(upper=20)
    axes[1].hist(lead, bins=np.arange(1, 22), density=True, histtype="step", linewidth=1.8,
                 color=CITY_COLOR[city])
axes[1].set_xlabel("lead over the second cell, in weekdays (20 = 20+)", fontsize=FS2)
axes[1].set_ylabel("share of work users", fontsize=FS2)
style_axis(axes[1], "(B)")
plt.tight_layout()

---
# 2. Commute distance

Distance between the centres of the work cell and the home cell (res 10, or res 9 for res 9-only homes). Work
cells start at ~200 m, since the home area is excluded. Log scale.

In [ ]:
# ── plot: distance distribution per city, median marked ───────────────────────
fig, ax = plt.subplots(figsize=(8, 4.2))
bins = np.logspace(np.log10(150), np.log10(60000), 50)
for city in CITIES:
    d = wdays.loc[wdays["city"] == city, "dist_home_m"]
    ax.hist(d, bins=bins, density=False, weights=np.full(len(d), 1 / len(d)), histtype="step",
            linewidth=1.8, color=CITY_COLOR[city], label=f"{CITY_LABEL[city]} (median {d.median() / 1000:.1f} km)")
    ax.axvline(d.median(), color=CITY_COLOR[city], linewidth=1, linestyle=":")
ax.set_xscale("log")
ax.set_xticks([200, 500, 1000, 2000, 5000, 10000, 20000, 50000])
ax.set_xticklabels(["0.2", "0.5", "1", "2", "5", "10", "20", "50"])
ax.set_xlabel("home to work distance (km)", fontsize=FS2)
ax.set_ylabel("share of work users", fontsize=FS2)
ax.legend(frameon=False, fontsize=FS2)
style_axis(ax)
plt.tight_layout()

---
# 3. Maps
## 3.1 Work places per res 9 cell

Work places aggregated to res 9 (res 10 is too sparse to read). Log colour scale.

In [ ]:
# ── work places per res 9 cell ────────────────────────────────────────────────
wmap = read_hex(f"""
select city, public.h3_cell_to_parent(cell, 9) as cell, count(*) as n_work
from {WORK} w
group by 1, 2
""")

# ── plot: one panel per city ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
for ax, city, panel in zip(axes, CITIES, PANEL):
    g = wmap[wmap["city"] == city]
    hex_map(ax, g, "n_work", LogNorm(1, g["n_work"].max()), "work places per H3 res 9 cell",
            f"{panel} {CITY_LABEL[city]}", cmap="Oranges")
plt.tight_layout()

## 3.2 Where work users live vs where they work (London centre)

Same users in both panels: home cells (res 9) of users with a work place, and their work cells (res 9).

In [ ]:
# ── homes and work places of work users, London, res 9 ────────────────────────
lw = read_hex(f"""
select 'home' as kind, public.h3_cell_to_parent(h.cell, 9) as cell, count(*) as n
from {WORK} w
join revised.home_final h
  on h.city = w.city and h.user_id = w.user_id and h.res = 9
where w.city = 'london'
group by 1, 2
union all
select 'work', public.h3_cell_to_parent(cell, 9), count(*)
from {WORK} w
where city = 'london'
group by 1, 2
""")

# ── plot: same colour scale in both panels ────────────────────────────────────
norm = LogNorm(1, lw["n"].max())
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
hex_map(axes[0], lw[lw["kind"] == "home"], "n", norm, "work users living in the res 9 cell",
        "(A) homes", extent=zoom_extent("london"))
hex_map(axes[1], lw[lw["kind"] == "work"], "n", norm, "work users working in the res 9 cell",
        "(B) work places", extent=zoom_extent("london"), cmap="Oranges")
plt.tight_layout()

## 3.3 Top work cells

Expect business districts, universities, stations and airports. Soho (London) holds a neighbourhood check-in point
used by ~16,500 users, which inflates its count.

In [ ]:
# ── top 10 work cells per city ────────────────────────────────────────────────
pd.read_sql(f"""
select city, cell::text as cell, n_users, lat, lon
from (
    select city
         , cell
         , count(*)                                                       as n_users
         , round(public.st_y(public.h3_cell_to_geometry(cell))::numeric, 5) as lat
         , round(public.st_x(public.h3_cell_to_geometry(cell))::numeric, 5) as lon
         , row_number() over (partition by city order by count(*) desc)   as rk
    from {WORK} w
    group by city, cell
) x
where rk <= 10
order by city, rk
""", engine)

---
# 4. Timing: when are people at home, at work, elsewhere?

Stays (`visit_stay`: user, day, res 10 cell) of work users, labelled home area / work cell / other, by the local
hour of the stay's first tweet. Weekdays solid, weekends dashed; each line sums to 1. Work stays should peak in
office hours on weekdays and nearly vanish at weekends; home stays should peak in the evening.

In [ ]:
# ── stays of work users by label, weekday / weekend and hour ──────────────────
timing = pd.read_sql(f"""
select s.city
     , case when a.cell_10 is not null then 'home'
            when s.cell_10 = w.cell    then 'work'
            else 'other'
       end                                                 as label
     , s.dow <= 5                                          as weekday
     , extract(hour from s.first_local)::int               as hour
     , count(*)                                            as n
from revised.visit_stay s
join {WORK} w
  on w.city = s.city and w.user_id = s.user_id
left join revised.visit_home_area a
  on a.city = s.city and a.user_id = s.user_id and a.cell_10 = s.cell_10
group by 1, 2, 3, 4
""", engine)
timing["share"] = timing["n"] / timing.groupby(["city", "label", "weekday"])["n"].transform("sum")

# ── plot: one panel per city ──────────────────────────────────────────────────
LABEL_COLOR = {"home": "#2a78d6", "work": "#eb6834", "other": "0.5"}
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2), sharey=True)
for ax, city, panel in zip(axes, CITIES, PANEL):
    for label, color in LABEL_COLOR.items():
        for wd, ls in [(True, "-"), (False, "--")]:
            d = timing[(timing["city"] == city) & (timing["label"] == label)
                       & (timing["weekday"] == wd)].sort_values("hour")
            ax.plot(d["hour"], d["share"], color=color, linestyle=ls, linewidth=1.6)
    ax.set_xticks(range(0, 24, 3))
    ax.set_xlabel("local hour of the first tweet of the stay", fontsize=FS2)
    style_axis(ax, f"{panel} {CITY_LABEL[city]}")
axes[0].set_ylabel("share of stays", fontsize=FS2)
handles = [Line2D([], [], color=c, label=l) for l, c in LABEL_COLOR.items()]
handles += [Line2D([], [], color="0.2", linestyle="-", label="Mon-Fri"),
            Line2D([], [], color="0.2", linestyle="--", label="Sat-Sun")]
axes[2].legend(handles=handles, frameon=False, fontsize=FS2)
plt.tight_layout()